# Preprocessing

In [39]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
import joblib
import numpy as np

In [22]:
train_test_size = 0.3 # size of test dataset on training split
val_test_size = 0.5 # size of test dataset on validation split

## Feature Selection, Strings and Methods definition

In [23]:
df = pd.read_csv("../data/bank_churn_raw.csv")

In [40]:
target_column = "Exited"
columns_to_drop = ["CustomerId", "Surname"]

cat_cols = ["Geography", "Gender", "IsActiveMember", "HasCrCard"]

# classification or regression
output_type = "classification"

scaler = StandardScaler()

In [25]:
df = df.drop(columns=columns_to_drop)

In [26]:
num_cols = [col for col in df.columns if (col not in cat_cols and col != target_column)]

In [27]:
X, y = df.drop(columns=[target_column]), df[target_column]

## Train Test Split

In [28]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=train_test_size, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=val_test_size, random_state=42)

## Handle missing values

In [29]:
df.isnull().sum()

CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

## Categorical Encoding

In [30]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

encoder.fit(X_train[cat_cols])

X_train_encoded = encoder.transform(X_train[cat_cols])
X_val_encoded = encoder.transform(X_val[cat_cols])
X_test_encoded = encoder.transform(X_test[cat_cols])

In [31]:
encoded_cols = encoder.get_feature_names_out(cat_cols)

X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=encoded_cols, index=X_train.index)
X_val_encoded_df   = pd.DataFrame(X_val_encoded,   columns=encoded_cols, index=X_val.index)
X_test_encoded_df  = pd.DataFrame(X_test_encoded,  columns=encoded_cols, index=X_test.index)

In [32]:
# Encoding Sanity Check
for col in cat_cols:
    col_encoded = [c for c in encoded_cols if c.startswith(col + "_")]

    temp = pd.concat([X_train[[col]], X_train_encoded_df[col_encoded]], axis=1)

    # For each original category, check how many unique encoded rows exist
    check = temp.groupby(col)[col_encoded].nunique()

    if (check > 1).any().any():
        print(f"Problem in column: {col}")
    else:
        print(f"Column OK: {col}")

Column OK: Geography
Column OK: Gender
Column OK: IsActiveMember
Column OK: HasCrCard


In [33]:
X_train = pd.concat([X_train, X_train_encoded_df], axis=1)
X_val   = pd.concat([X_val, X_val_encoded_df], axis=1)
X_test  = pd.concat([X_test, X_test_encoded_df], axis=1)

X_train = X_train.drop(columns=cat_cols)
X_val   = X_val.drop(columns=cat_cols)
X_test  = X_test.drop(columns=cat_cols)

## Scaling

In [34]:
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

X_val[num_cols]  = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

## Save fitting parameters

In [38]:
joblib.dump(scaler, '../output/models/scaler.pkl')
joblib.dump(encoder, '../output/models/encoder.pkl')

['../output/models/encoder.pkl']

## Final Sanity Check

In [41]:
# No missing values remain
assert X_train.isnull().sum().sum() == 0

# Shapes are consistent
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]

# No infinite values
assert np.isfinite(X_train.values).all()

# Target distribution looks reasonable
if(output_type == "classification"):
    print(y_train.value_counts(normalize=True))
elif(output_type == "regression"):
    print(y_train.describe())

Exited
0    0.792429
1    0.207571
Name: proportion, dtype: float64


## Save Data

In [44]:
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/X_val.npy', X_val)
np.save('../data/processed/X_test.npy', X_test)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_val.npy', y_val)
np.save('../data/processed/y_test.npy', y_test)